# Transformation 00 — Target Binarization


In [26]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from helpers.data_loader import DataLoader

train = pd.read_parquet(DataLoader.processed('train.parquet'))
test  = pd.read_parquet(DataLoader.processed('test.parquet'))
print('Train shape:', train.shape)
print('Test  shape:', test.shape)

Train shape: (137176, 26)
Test  shape: (34294, 26)


## 1 · Class distribution BEFORE binarization

In [27]:
print('=== Train — Results distribution (before) ===')
print(train['Results'].value_counts(normalize=True).mul(100).round(2))
print()
print('=== Test — Results distribution (before) ===')
print(test['Results'].value_counts(normalize=True).mul(100).round(2))

=== Train — Results distribution (before) ===
Results
Pass                  67.06
Fail                   22.1
Pass w/ Conditions    10.84
Name: proportion, dtype: Float64

=== Test — Results distribution (before) ===
Results
Pass                  40.86
Pass w/ Conditions    36.58
Fail                  22.55
Name: proportion, dtype: Float64


## 2 · Binarize the target

In [28]:
# -- Violation count (used to split PwC rows) --
def count_violations(series: pd.Series) -> pd.Series:
    return (
        series
        .fillna('')
        .apply(lambda x: len(x.split(' | ')) if x.strip() else 0)
    )

train['violation_count'] = count_violations(train['Violations'])
test['violation_count']  = count_violations(test['Violations'])

# Explore threshold on train PwC rows only (no test leakage)
pwc_train = train[train['Results'] == 'Pass w/ Conditions']
print('PwC violation_count distribution:')
print(pwc_train['violation_count'].describe())
print()
for t in range(1, 8):
    frac_pass = (pwc_train['violation_count'] < t).mean()
    print(f'  threshold={t}:  {frac_pass:.1%} → Pass,  {1-frac_pass:.1%} → Fail')

PwC violation_count distribution:
count    14866.000000
mean         4.952576
std          2.510892
min          0.000000
25%          3.000000
50%          5.000000
75%          6.000000
max         23.000000
Name: violation_count, dtype: float64

  threshold=1:  0.9% → Pass,  99.1% → Fail
  threshold=2:  6.3% → Pass,  93.7% → Fail
  threshold=3:  15.9% → Pass,  84.1% → Fail
  threshold=4:  30.5% → Pass,  69.5% → Fail
  threshold=5:  47.3% → Pass,  52.7% → Fail
  threshold=6:  62.8% → Pass,  37.2% → Fail
  threshold=7:  75.5% → Pass,  24.5% → Fail


In [ ]:
PwC_THRESHOLD = 4 

TARGET_MAP = {'Pass': 0, 'Fail': 1}

def binarize_target(df):
    binary = df['Results'].map(TARGET_MAP)
    pwc_mask = df['Results'] == 'Pass w/ Conditions'
    binary[pwc_mask] = (df.loc[pwc_mask, 'violation_count'] >= PwC_THRESHOLD).astype(int)
    assert binary.isna().sum() == 0, 'Unmapped values!'
    return binary.astype(int)

train['Results'] = binarize_target(train)
test['Results']  = binarize_target(test)

# train.drop(columns=['violation_count'], inplace=True)
# test.drop(columns=['violation_count'],  inplace=True)

print(f'Binarization complete (PwC threshold = {PwC_THRESHOLD}).')

Binarization complete (PwC threshold = 4).


## 3 · Class distribution AFTER binarization

In [30]:
print('=== Train — Results distribution (after) ===')
print(train['Results'].value_counts(normalize=True).mul(100).round(2))
print(f'Class ratio (0:1): {(train["Results"]==0).sum() / (train["Results"]==1).sum():.2f}:1')
print()
print('=== Test — Results distribution (after) ===')
print(test['Results'].value_counts(normalize=True).mul(100).round(2))
print(f'Class ratio (0:1): {(test["Results"]==0).sum() / (test["Results"]==1).sum():.2f}:1')

=== Train — Results distribution (after) ===
Results
0    70.36
1    29.64
Name: proportion, dtype: float64
Class ratio (0:1): 2.37:1

=== Test — Results distribution (after) ===
Results
0    51.37
1    48.63
Name: proportion, dtype: float64
Class ratio (0:1): 1.06:1


## 4 · Save intermediate results

In [31]:
train.to_parquet(DataLoader.transformed('train_t00.parquet'), index=False)
test.to_parquet(DataLoader.transformed('test_t00.parquet'), index=False)
print('Saved train_t00.parquet and test_t00.parquet to', DataLoader.TRANSFORMED_DIR)

Saved train_t00.parquet and test_t00.parquet to ..\..\data\transformed
